Excellent. Now we're entering one of the **most important LangGraph concepts**.

Many beginners understand **State, Nodes, and Edges**, but get confused about **Reducers**. Once you understand reducers, concepts like **MessagesState**, **parallel execution**, and **multi-agent graphs** become much easier.

---

# LangGraph Topic 5 — Reducers (Complete Beginner → Advanced)

**Official docs:** [LangGraph Reducers & State docs](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com#reducers)

---

# 1. Why Do We Need Reducers?

Let's start with a simple question.

Suppose your graph state is:

```python
class State(TypedDict):
    count: int
```

Current state:

```python
{
    "count": 5
}
```

Node returns:

```python
{
    "count": 10
}
```

How should LangGraph update the state?

Option 1

```text
Replace 5 with 10
```

Option 2

```text
5 + 10 = 15
```

Option 3

```text
Keep both values
```

**Who decides this?**

➡️ **Reducers decide how state updates are merged.**

---

# 2. What is a Reducer?

A **Reducer** is a function that tells LangGraph:

> **"When a node returns a value for a state key, how should that value be merged with the existing value?"**

Without reducers, LangGraph wouldn't know whether to:

* replace
* append
* merge
* add
* concatenate
* perform custom logic

---

# 3. Default Behavior (Without a Reducer)

Suppose:

```python
class State(TypedDict):
    answer: str
```

Current state:

```python
{
    "answer": "Hello"
}
```

Node returns

```python
{
    "answer": "Hi"
}
```

Final state becomes

```python
{
    "answer": "Hi"
}
```

The old value is **replaced**.

This is LangGraph's **default reducer**.

Think of it as:

```python
new_value = update
```

---

# 4. Visualizing the Default Reducer

```text
Current State

answer = "Hello"

        │
        ▼

Node returns

answer = "Hi"

        │
        ▼

Reducer

Replace old value

        │
        ▼

answer = "Hi"
```

---

# 5. Why Replacement Isn't Always Enough

Imagine a chat application.

State:

```python
messages = [
    "Hello"
]
```

Assistant replies

```python
messages = [
    "How are you?"
]
```

Should we replace the old messages?

If we do:

```python
[
    "How are you?"
]
```

We lose the conversation history.

Instead we want:

```python
[
    "Hello",
    "How are you?"
]
```

So for chat history we need:

> **Append instead of Replace**

That's where reducers become essential.

---

# 6. Think of Reducers Like `list.append()`

Imagine this Python code:

```python
numbers = [1, 2]

numbers.append(3)
```

Result

```python
[1, 2, 3]
```

A reducer can implement this same behavior automatically whenever a node returns new values.

---

# 7. A Reducer is Just a Python Function

Example:

```python
def add_numbers(current, new):
    return current + new
```

Suppose

Current value

```python
[1, 2]
```

Node returns

```python
[3]
```

Reducer executes

```python
[1,2] + [3]
```

Result

```python
[1,2,3]
```

---

# 8. Where Are Reducers Defined?

Reducers are attached to **individual state fields**, not to the whole graph.

Example:

```python
class State(TypedDict):
    messages: ?
```

Instead of only specifying the type, we also specify **how updates are merged**.

LangGraph uses Python's `Annotated` type for this.

---

# 9. Introducing `Annotated`

Basic state

```python
class State(TypedDict):
    messages: list
```

State with reducer

```python
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list, reducer_function]
```

General syntax

```python
field_name: Annotated[type, reducer]
```

---

# 10. Example Using `operator.add`

Python already has a built-in function:

```python
from operator import add
```

Example

```python
add([1,2], [3])
```

Output

```python
[1,2,3]
```

LangGraph can use that as a reducer.

---

# 11. Full Example

```python
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    numbers: Annotated[list[int], add]
```

Current state

```python
{
    "numbers": [1,2]
}
```

Node returns

```python
{
    "numbers": [3]
}
```

Reducer executes

```python
add([1,2], [3])
```

Result

```python
{
    "numbers":[1,2,3]
}
```

---

# 12. Reducer Execution Flow

```text
Current State

numbers = [1,2]

        │

Node returns

numbers = [3]

        │

Reducer

add(old,new)

        │

Final

numbers = [1,2,3]
```

---

# 13. Another Reducer Example

Suppose we want integers to accumulate.

Reducer

```python
from operator import add
```

State

```python
class State(TypedDict):
    total: Annotated[int, add]
```

Current state

```python
10
```

Node returns

```python
5
```

Reducer

```python
10 + 5
```

Final

```python
15
```

---

# 14. Why Reducers Become Important in Parallel Execution

Imagine this graph

```text
          START
             │
             ▼
          Node A
          /     \
         ▼       ▼
     Node B   Node C
         \       /
          ▼     ▼
           END
```

Both B and C update

```python
messages
```

Without reducers...

Who wins?

B?

C?

Both?

LangGraph needs rules.

Reducers solve this merge problem.

---

# 15. Parallel Example

Current state

```python
messages=[]
```

Node B returns

```python
["Hello"]
```

Node C returns

```python
["World"]
```

Reducer (`operator.add`)

Result

```python
["Hello","World"]
```

Without reducer

Only one update might survive.

---

# 16. Why Reducers Matter in LLM Applications

Consider a RAG system.

Node A retrieves

```python
["Doc1"]
```

Node B retrieves

```python
["Doc2"]
```

You want

```python
["Doc1","Doc2"]
```

Not

```python
["Doc2"]
```

Reducers preserve all retrieved documents.

---

# 17. Custom Reducers

Reducers don't have to use `operator.add`.

You can write your own.

Example

```python
def keep_longer(old, new):
    if len(new) > len(old):
        return new
    return old
```

State

```python
summary: Annotated[str, keep_longer]
```

Now LangGraph always keeps the longer summary.

---

# 18. Another Custom Reducer

Suppose we always want the maximum score.

```python
def maximum(old, new):
    return max(old, new)
```

Current

```python
80
```

Update

```python
95
```

Final

```python
95
```

---

# 19. MessagesState Uses Reducers

This is one reason `MessagesState` exists.

A chatbot shouldn't replace messages.

Instead

```
User message

↓

Assistant message

↓

Tool message

↓

Assistant message
```

should accumulate.

Internally, `MessagesState` uses a specialized reducer (commonly `add_messages`) to merge message history intelligently rather than replacing it.

---

# 20. Default Replace vs Custom Reducer

Without reducer

```text
Old

["A"]

↓

Node

["B"]

↓

Final

["B"]
```

With reducer (`add`)

```text
Old

["A"]

↓

Node

["B"]

↓

Final

["A","B"]
```

---

# 21. Common Reducers You'll See

| Reducer        | Purpose                          |
| -------------- | -------------------------------- |
| Default        | Replace old value                |
| `operator.add` | Add integers / concatenate lists |
| `add_messages` | Merge chat messages              |
| Custom reducer | Any merge logic you define       |

---

# 22. Real AI Example

State

```python
class State(TypedDict):
    retrieved_docs: Annotated[list[str], add]
```

Three retrieval nodes run

Node 1

```python
["Python Guide"]
```

Node 2

```python
["LangGraph Tutorial"]
```

Node 3

```python
["LLM Paper"]
```

Final state

```python
[
 "Python Guide",
 "LangGraph Tutorial",
 "LLM Paper"
]
```

---

# 23. When Do You Need a Reducer?

You usually need one when:

* multiple nodes update the same key
* you want to preserve history
* you're storing chat messages
* you're accumulating results
* your graph has parallel branches

For simple values like:

```python
answer: str
```

the default replace behavior is often sufficient.

---

# 24. Common Beginner Mistakes

### Mistake 1

Expecting lists to append automatically.

They won't unless a reducer is specified.

---

### Mistake 2

Using `operator.add` on data where replacement is intended.

Not every field should accumulate.

---

### Mistake 3

Forgetting that reducers operate **per field**, not for the whole state.

---

### Mistake 4

Writing reducers that mutate the existing object instead of returning a new merged value. Prefer returning the merged result.

---

# 25. Interview Questions

1. What is a reducer in LangGraph?
2. What is the default reducer behavior?
3. Why are reducers important for parallel execution?
4. Why does `MessagesState` need a reducer?
5. What is the role of `Annotated`?
6. When would you use `operator.add`?
7. Can you write a custom reducer?
8. What happens if two nodes update the same key?

---

# 26. Cheat Sheet

```text
REDUCERS

Purpose
-------
Decide how state updates are merged.

Default
-------
Replace old value.

Syntax
------
Annotated[type, reducer]

Examples
--------
operator.add
add_messages
custom reducer

Use Cases
---------
Chat history
Parallel nodes
Document retrieval
Counters
Lists
```

# 27. Practice Exercise

Create this state:

```python
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    logs: Annotated[list[str], add]
```

Write two nodes:

```python
def node_a(state):
    return {"logs": ["Started processing"]}

def node_b(state):
    return {"logs": ["Finished processing"]}
```

Connect them:

```text
START → node_a → node_b → END
```

Answer: `["Started processing", "Finished processing"]`

In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    logs: Annotated[list[str], add]


def node_a(state):
    return {"logs": ["Started processing"]}


def node_b(state):
    return {"logs": ["Finished processing"]}


builder = StateGraph(State)

builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()

result = graph.invoke({"logs": []})

print(result["logs"])

['Started processing', 'Finished processing']
